<a href="https://colab.research.google.com/github/Kownlage/English/blob/main/quicksort_log_analysis_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Quick Sort Applications and Comparison

**Course:** CSCI 232 — Data Structures\
**Duration:** 1 Week\
**Total:** 100 Points

Reuse your quick sort to find percentiles and the largest values of an array **without
sorting the whole array**, then compare quick sort and merge sort on memory and stability.

After you paste your code, this notebook runs independently in Colab or locally.
Unfinished functions report PENDING rather than stopping Run All.

## Workflow

| Part | What you do | Points |
|---|---|---|
| Part 1 | Paste your `quick_sort.py` and `merge_sort.py` | Required, 0 points |
| Part 2 | Write `quickselect`, `percentile`, `top_k` and their tests; watch your code in the replay | 60 |
| Part 3 | Run the provided helpers, write `check_stability`, explain the results | 40 |

Run the check cell after editing any function. Every group must pass:
**ALL CHECKS PASSED**.

### Rules

- Keep every given name, signature and docstring. A docstring states the cost and
  whether the input changes.
- Do not use `sorted()`, `list.sort()`, `heapq`, `numpy` or `statistics` in your
  Part 1 and Part 2 functions. Your tests in Task 2.4 may use `sorted()` as the
  reference answer.
- A function must not change its input, unless its docstring says it works in place.

In [175]:
import gc
import inspect
import math
import random
import time
import tracemalloc
from dataclasses import dataclass
from fractions import Fraction
from html import escape
from typing import List, Sequence, TypeVar

T = TypeVar('T')


def _placeholder(name):
    """Stand in for a function not pasted yet, so that Run All reports PENDING."""
    def unfinished(*args, **kwargs):
        raise NotImplementedError(name)
    unfinished.__name__ = name
    return unfinished


for _name in ('quick_sort', '_quick_sort_range', '_partition',
              'merge_sort', '_merge_sort_range', '_merge'):
    globals().setdefault(_name, _placeholder(_name))  # your pasted code replaces these
print('Setup complete. Continue from top to bottom.')

Setup complete. Continue from top to bottom.


## Part 1 — Paste your quick sort and merge sort

Paste your own completed code into the two empty cells below. Do not rewrite it or change
it: Part 2 calls these functions exactly as you pasted them.

| Cell | Paste the full contents of | Functions it defines |
|---|---|---|
| First | `dsa/sorting/quick_sort.py` | `quick_sort`, `_quick_sort_range`, `_partition` |
| Second | `dsa/sorting/merge_sort.py` | `merge_sort`, `_merge_sort_range`, `_merge` |

Part 1 earns no points, but Parts 2 and 3 use these functions.

In [176]:
"""Quick sort algorithm."""

from typing import TypeVar, List

T = TypeVar('T')


def quick_sort(data: List[T]) -> None:
    """Sort the list in place using quick sort.

    Quick sort is a divide-and-conquer algorithm that selects a pivot
    element, partitions the list around the pivot (elements less than
    pivot on left, greater on right), and recursively sorts the partitions.

    Time complexity: O(n log n) average case, O(n^2) worst case
    Space complexity: O(log n) average case for recursion stack
    Stable: No

    Args:
        data: The list to sort in place.
    """
    if len(data) > 1:
        _quick_sort_range(data, 0, len(data) - 1)


def _quick_sort_range(data: List[T], low: int, high: int) -> None:
    """Sort the subarray data[low:high+1] using quick sort.

    Args:
        data: The list containing the subarray.
        low: Starting index of the subarray.
        high: Ending index of the subarray.
    """
    if low >= high:
        return

    p = _partition(data, low, high)
    _quick_sort_range(data, low, p - 1)
    _quick_sort_range(data, p + 1, high)


def _partition(data: List[T], low: int, high: int) -> int:
    """Partition the subarray around a pivot.

    Rearranges elements so that all elements less than the pivot come
    before it, and all elements greater come after it.

    Args:
        data: The list containing the subarray.
        low: Starting index of the subarray.
        high: Ending index of the subarray.

    Returns:
        The final index of the pivot element.
    """
    pivot = data[high]
    left = low
    right = high - 1

    while True:
        while left <= right and data[left] < pivot:
            left += 1
        while left <= right and data[right] > pivot:
            right -= 1

        if left <= right:
            data[left], data[right] = data[right], data[left]
            left += 1
            right -= 1
        else:
            break

    data[left], data[high] = data[high], data[left]
    return left

In [177]:
"""Merge sort algorithm."""

from typing import TypeVar, List

T = TypeVar('T')


def merge_sort(data: List[T]) -> None:
    """Sort the list in place using merge sort.

    Merge sort is a divide-and-conquer algorithm that divides the list
    into halves, recursively sorts each half, and then merges the sorted
    halves back together.

    Time complexity: O(n log n) in all cases
    Space complexity: O(n) for the auxiliary array
    Stable: Yes

    Args:
        data: The list to sort in place.
    """
    n = len(data)
    if n <= 1:
        return
    _merge_sort_range(data, 0, n - 1)


def _merge_sort_range(data: List[T], left: int, right: int) -> None:
    if left >= right:
        return
    mid = (left + right) // 2
    _merge_sort_range(data, left, mid)
    _merge_sort_range(data, mid + 1, right)
    _merge(data, left, mid, right)


def _merge(data: List[T], left: int, mid: int, right: int) -> None:
    """Merge two sorted subarrays data[left:mid+1] and data[mid+1:right+1].

    Args:
        data: The list containing the subarrays.
        left: Starting index of the first subarray.
        mid: Ending index of the first subarray.
        right: Ending index of the second subarray.
    """
    left_copy = data[left:mid + 1]
    right_copy = data[mid + 1:right + 1]

    i = 0
    j = 0
    k = left

    while i < len(left_copy) and j < len(right_copy):
        if left_copy[i] <= right_copy[j]:
            data[k] = left_copy[i]
            i += 1
        else:
            data[k] = right_copy[j]
            j += 1
        k += 1
    while i < len(left_copy):
        data[k] = left_copy[i]
        i += 1
        k += 1
    while j < len(right_copy):
        data[k] = right_copy[j]
        j += 1
        k += 1


## Part 2 — Percentile Using Quickselect (60 Points)

### Scenario: Web Server Response Times

A web server needs to calculate the **90th percentile (P90)** of response times
without sorting the entire dataset. P90 = 310 ms would mean that 90% of requests
took 310 ms or less.

**Example:**

```python
data = [120, 45, 310, 80, 200, 95, 410, 60, 150, 240]
# Expected P90: 310 ms
```

### Tasks

| Function | Changes its input? | Invalid input |
|---|---|---|
| `quickselect(data, k)` | **Yes**, it rearranges `data` in place | `IndexError` unless `0 <= k < len(data)`; an empty list always raises |
| `percentile(data, p)` | No, it works on a copy | `ValueError` if `data` is empty, `p <= 0` or `p > 100` |
| `top_k(data, k)` | No, it works on a copy | None: `k <= 0` or empty `data` returns `[]`; `k >= n` returns everything, largest first |

The next cell provides `nearest_rank_index`, which computes the percentile index with
exact arithmetic. Do not edit it.

In [178]:
def nearest_rank_index(p: float, n: int) -> int:
    """Provided. Return the 0-based nearest-rank index, ceil(p / 100 * n) - 1. O(1).

    The arithmetic is exact: with floats, ceil(7 / 100 * 100) is 8, not 7.
    numpy.percentile interpolates linearly by default, so it can differ from
    nearest-rank. That difference comes from the definition; it is not a bug.
    This function does not validate p or n; percentile() does that.
    """
    return math.ceil(Fraction(str(p)) * n / 100) - 1

**2.1 Implement Quickselect (25 pts)**

Requirements:

- Reuse `_partition()` from Part 1 exactly as you pasted it. Do not change or redefine it.
- Process only the partition containing index `k`. Use a loop, not recursion.
- Do not fully sort the array.
- Before each partition, swap a random slot, `random.randint(low, high)`, into slot
  `high`. Without this step, input that is already sorted makes every pivot the
  largest value in its range, and the check on 1,000,000 sorted values would not finish.
- `quickselect` **rearranges `data` in place**, so a caller who needs the original
  order must pass a copy. When it returns, `data[:k]` is at most `data[k]`, and
  `data[k+1:]` is at least `data[k]`.

In [179]:
def quickselect(data: List[T], k: int) -> T:
    """Return the element at sorted index k (0-based).
    Works IN PLACE on `data` (rearranges it). Average O(n), worst O(n^2).
    After return: every element in data[:k] is <= data[k],
    and every element in data[k+1:] is >= data[k].
    Raises:
        IndexError: unless 0 <= k < len(data); an empty list always raises.
    """
    if not (0 <= k < len(data)):
        raise IndexError("k is out of bounds for the given data list.")

    low = 0
    high = len(data) - 1

    while low < high:

        rand_idx = random.randint(low, high)
        data[rand_idx], data[high] = data[high], data[rand_idx]

        p = _partition(data, low, high)

        if p == k:
            return data[k]
        elif p < k:
            low = p + 1
        else:
            high = p - 1

    return data[k]

**2.2 Calculate Percentile (15 pts)**

Use the nearest-rank method. For n values sorted ascending, the answer is at

```text
k = ceil(p / 100 * n) - 1      # 0-based index
```

Requirements:

- Use Quickselect to calculate P90. Find the index with `nearest_rank_index`, because
  the formula is unsafe in floating point.
- Preserve the original input array.
- Handle invalid inputs: raise `ValueError` for empty data and for `p` outside (0, 100].

In [180]:
def percentile(data: Sequence[T], p: float) -> T:
    """Nearest-rank p-th percentile. Does NOT modify `data` (works on a copy).
    Average O(n), worst O(n^2).
    Raises:
        ValueError: if data is empty, or p is outside (0, 100].
    """
    if not data or not (0 < p <= 100):
        raise ValueError("Data cannot be empty and p must be in the range (0, 100].")

    data_copy = list(data)

    k = nearest_rank_index(p, len(data_copy))

    return quickselect(data_copy, k)

**2.3 Find the Top k (10 pts)**

The team also wants the k slowest response times, largest first. `top_k` needs only one
extra idea. `quickselect(copy, n - k)` leaves the k largest values in `copy[n - k:]`,
unordered, so only that suffix needs sorting.

Requirements:

- Follow the three steps in the docstring, using your `quick_sort` from Part 1 for step 3.
- Preserve the original input array.
- Handle `k <= 0`, `k >= n` and empty data as the table above says.

Example: `top_k([5, 1, 9, 7], 2)` returns `[9, 7]`.

*Limit:* your `quick_sort` always takes the last element as its pivot. If the top values
happen to be in order and k is in the thousands, step 3 can exceed Python's recursion
limit and raise `RecursionError`. This lab only uses small k, such as 10.

In [181]:
def top_k(data: Sequence[T], k: int) -> list[T]:
    """Return the k largest elements, ordered from largest to smallest.
    Algorithm:
      1. copy = list(data)
      2. quickselect(copy, n - k)  -> copy[n-k:] holds the top k (unordered)
      3. result = copy[n-k:]; quick_sort(result); result.reverse()
    Average O(n + k log k). Does NOT modify `data`.
    k <= 0 or empty data gives []; k >= n gives all of data, largest first.
    """
    n = len(data)

    if not data or k <= 0:
        return []
    if k >= n:
        sorted_data = list(data)
        quick_sort(sorted_data)
        sorted_data.reverse()
        return sorted_data
    copy = list(data)

    quickselect(copy, n - k)

    result = copy[n - k:]
    quick_sort(result)
    result.reverse()

    return result

**Replay: watch your code run (no points)**

Run the next two cells after you finish 2.1 and 2.3. The replay calls **your own**
`quickselect` and `top_k` on a 10-value array and records every comparison and swap, so
you can see how the array moves, and a bug shows up in the picture. Optional `ipywidgets`
gives the controls. The second cell shows the last step of two runs without controls.

1. Choose a function, an input and `k`. For `quickselect`, `k` is a sorted index. For
   `top_k`, it is how many values you want.
2. Step with **Previous step**, **Next step** and **Final step**, or drag the slider.
3. After changing your code, rerun its cell and click **Run again**. It calls your current
   code with new random pivots.

Things to look for (no written answer is required):

- `quickselect` on **P90 example** with k = 8: how fast does the navy range shrink? How
  many slots would quick sort still have to visit?
- `top_k` on **P90 example** with k = 3: which slots does `quick_sort(result)` touch, and
  which slots are never touched again?
- `quickselect` on **All equal**: where does each pivot land, and why does that help?

Without widgets, run `show_replay('top_k', 'P90 example', 3, step=0)` in a new cell and
change `step`. The second argument can also be your own list of at most 16 values.

In [182]:
REPLAY_SAMPLES = {                 # 10 values, so that a replay stays readable
    'P90 example': [120, 45, 310, 80, 200, 95, 410, 60, 150, 240],
    'Already sorted': [45, 60, 80, 95, 120, 150, 200, 240, 310, 410],
    'Reverse-sorted': [410, 310, 240, 200, 150, 120, 95, 80, 60, 45],
    'Many duplicates': [200, 80, 200, 45, 80, 200, 310, 45, 80, 120],
    'All equal': [100] * 10,
}
REPLAY_DEFAULT_K = {'quickselect': 8, 'top_k': 3}
REPLAY_SEED = 232
REPLAY_MAX_VALUES = 16
REPLAY_MAX_FRAMES = 5_000          # more steps than this means a loop that never ends
SLOT_COLORS = {'answer': '#1baf7a', 'pivot': '#eb6834', 'active': '#31475d', 'out': '#d7dde3'}
OUTLINE_COLORS = {'compare': '#b37a15', 'swap': '#2a78d6'}
REPLAY_CSS = """
<style>
.qs-glass{font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Arial,sans-serif;color:#17202a;
 background:radial-gradient(ellipse at top left,#fff,transparent 65%),linear-gradient(135deg,#edf0f3,#dce2e8);
 border:1px solid #fff;border-radius:22px;padding:18px 20px;box-shadow:0 12px 32px #16223320;
 line-height:1.45;max-width:1080px;margin-bottom:12px}
.qs-glass *{box-sizing:border-box}.qs-glass h2{font-size:21px;margin:0 0 6px;color:#17202a}
.qs-glass h3{font-size:15px;margin:0 0 8px;color:#17202a}
.qs-glass .chip{display:inline-block;background:#f7f8fa;border:1px solid #c4ccd4;border-radius:30px;
 padding:3px 10px;margin:2px 4px 4px 0;font-size:12px;font-weight:650}
.qs-glass .card{background:#ffffffd9;border:1px solid #fff;border-radius:14px;padding:12px 14px;
 margin:10px 0;box-shadow:0 6px 20px #17202a0a}
.qs-glass table{border-collapse:collapse;width:100%;font-size:13px;table-layout:auto !important}
.qs-glass th,.qs-glass td{padding:6px 8px;border-bottom:1px solid #d7dde3;text-align:left !important;
 vertical-align:middle;overflow-wrap:normal !important}
.qs-glass th{background:#f0f3f6;color:#24364b}
.qs-glass th.num,.qs-glass td.num{text-align:right !important;font-variant-numeric:tabular-nums;white-space:nowrap}
.qs-glass code{font-family:ui-monospace,SFMono-Regular,Consolas,monospace;font-size:.95em}
.qs-glass .muted{color:#5a6775;font-size:12px}
.qs-glass .banner{border-left:4px solid #31475d;padding:8px 12px;background:#eef3f7;border-radius:8px;
 font-size:13px;margin:8px 0}.qs-glass .banner.bad{border-color:#991c3d;background:#fff0f2}
.qs-glass .banner.warn{border-color:#9b6a16;background:#fff7e5}
.qs-glass svg{display:block;width:100%;height:auto}
.qs-glass .strip{white-space:nowrap}
.qs-glass .cell{display:inline-block;min-width:30px;padding:1px 3px;margin:0 1px;border-radius:4px;
 text-align:center;font-size:11px;font-variant-numeric:tabular-nums}
.qs-glass .cell.active{background:#31475d;color:#fff}.qs-glass .cell.pivot{background:#eb6834;color:#17202a}
.qs-glass .cell.answer{background:#1baf7a;color:#0b2b1f}.qs-glass .cell.out{background:#e4e8ec;color:#5a6775}
</style>
"""


class ReplayTooLong(Exception):
    """A replay recorded too many steps; its loop probably never ends."""


class replace_global:
    """Temporarily replace a notebook-level function; your code finds it by name."""

    def __init__(self, name, replacement):
        self.name, self.replacement = name, replacement

    def __enter__(self):
        self.saved = globals()[self.name]
        globals()[self.name] = self.replacement

    def __exit__(self, *exc_info):
        globals()[self.name] = self.saved


class TraceValue:
    """One array value that reports each comparison to the running replay."""
    __slots__ = ('value', 'origin')
    recorder = None

    def __init__(self, value, origin):
        self.value, self.origin = value, origin

    def __lt__(self, other):
        TraceValue.recorder.compare(self, other)
        return self.value < other.value

    def __le__(self, other):
        TraceValue.recorder.compare(self, other)
        return self.value <= other.value

    def __gt__(self, other):
        TraceValue.recorder.compare(self, other)
        return self.value > other.value

    def __ge__(self, other):
        TraceValue.recorder.compare(self, other)
        return self.value >= other.value

    def __eq__(self, other):
        TraceValue.recorder.compare(self, other)
        return self.value == other.value

    def __ne__(self, other):
        TraceValue.recorder.compare(self, other)
        return self.value != other.value

    __hash__ = None


class TraceList(list):
    """A list that reports each change to the running replay."""

    def __setitem__(self, index, value):
        changed = isinstance(index, slice) or self[index] is not value
        super().__setitem__(index, value)
        if changed:
            TraceValue.recorder.write(index)


class ReplayRecorder:
    """Collects one snapshot of the array per comparison, swap or partition."""

    def __init__(self, name, values, k):
        self.name, self.values, self.k, self.n = name, values, k, len(values)
        self.target = k if name == 'quickselect' else self.n - k
        # `data` is the list your code is changing; `fixed` holds the slots to its left.
        self.fixed, self.data = [], [TraceValue(value, i) for i, value in enumerate(values)]
        self.frames, self.rows, self.answer_slots = [], [], ()
        self.comparisons = self.swaps = self.rounds = 0
        self.low, self.high, self.pivot, self.phase = 0, self.n - 1, None, 'select'
        self.half_swap, self.partitioning, self.wrong = None, False, None

    def snap(self, kind, message, focus=()):
        if len(self.frames) >= REPLAY_MAX_FRAMES:
            raise ReplayTooLong
        self.frames.append({'kind': kind, 'message': message, 'focus': tuple(focus),
                            'order': [item.origin for item in [*self.fixed, *self.data]],
                            'comparisons': self.comparisons, 'swaps': self.swaps, 'rounds': self.rounds,
                            'phase': self.phase, 'low': self.low, 'high': self.high, 'pivot': self.pivot})

    def add_row(self, label, searched, size, pivot, landed, next_step):
        self.rows.append({'label': label, 'searched': searched, 'size': size, 'pivot': pivot,
                          'landed': landed, 'next': next_step, 'step': len(self.frames) - 1})

    def note(self, label, message):
        """One frame and one table row for a step that is not a partition."""
        self.snap('note', message)
        self.add_row(label, '—', '—', '—', '—', message)

    def compare(self, a, b):
        self.comparisons += 1
        shift = len(self.fixed)
        focus = [shift + slot for slot, item in enumerate(self.data) if item is a or item is b]
        self.snap('compare', f'Compare {a.value} with {b.value}', focus)

    def write(self, index):
        if isinstance(index, slice):
            self.snap('swap', 'Rewrite part of the list')
            return
        shift = len(self.fixed)
        slot = shift + index % len(self.data)
        if len({id(item) for item in self.data}) < len(self.data):
            self.half_swap = slot          # the first half of a swap: wait for the second
            return
        first = slot if self.half_swap is None else self.half_swap
        self.half_swap = None
        self.swaps += 1
        a, b = self.data[first - shift].value, self.data[slot - shift].value
        if self.partitioning:
            self.snap('swap', f'Swap {a} and {b} (slots {first} and {slot})', (first, slot))
        else:
            self.pivot = None
            self.snap('swap', f'Random pivot: swap {a} and {b} (slots {first} and {slot})', (first, slot))

    def open_round(self, low, high):
        shift = len(self.fixed)
        self.low, self.high, self.pivot = shift + low, shift + high, shift + high
        self.partitioning = True
        value = self.data[high].value
        if self.phase == 'sort':
            self.snap('round', f'Sort: partition slots {self.low}–{self.high} around {value}', [self.pivot])
        else:
            self.rounds += 1
            self.snap('round', f'Round {self.rounds}: search slots {self.low}–{self.high}; '
                               f'the pivot is {value} (slot {self.pivot})', [self.pivot])

    def close_round(self, index):
        self.partitioning = False
        slot = len(self.fixed) + index
        self.pivot = slot
        value = self.data[index].value
        if self.phase == 'sort':
            self.snap('landed', f'{value} is in its final place, slot {slot}', [slot])
            return
        if slot == self.target:
            side = 'stop: the pivot is on the target slot'
        elif self.target < slot:
            side = f'keep slots {self.low}–{slot - 1}'
        else:
            side = f'keep slots {slot + 1}–{self.high}'
        self.snap('landed', f'Pivot {value} landed on slot {slot}; {side}', [slot])
        self.add_row(f'Round {self.rounds}', f'slots {self.low}–{self.high}', self.high - self.low + 1,
                     value, slot, side)

    def after_select(self):
        self.phase, self.pivot = 'sort', None
        self.low = self.high = -1
        self.note('After quickselect', f'Slots {self.target}–{self.n - 1} hold the top {self.k}, '
                                       'in no particular order. The slots to their left are ruled out.')

    def start_sort(self, data):
        everything = [*self.fixed, *self.data]
        self.fixed, self.data = everything[:self.n - len(data)], TraceList(data)
        self.snap('note', f'quick_sort(result) sorts only these {len(data)} slots')

    def after_sort(self):
        self.pivot = None
        self.low = self.high = -1
        self.note('After quick_sort', 'The top values are sorted, smallest first.')

    def finish(self, label, message, slots):
        self.phase, self.pivot, self.answer_slots = 'done', None, tuple(slots)
        self.note(label, message)


def _replay_quickselect(recorder):
    """Run your quickselect directly on a list that reports every change."""
    k = recorder.k
    recorder.data = TraceList(recorder.data)
    answer = quickselect(recorder.data, k).value
    expected = sorted(recorder.values)[k]
    if answer != expected:
        recorder.wrong = f'Your quickselect returned {answer}, but sorted index {k} holds {expected}.'
    recorder.finish('Returned', f'Slot {k} holds {answer}. Values to its left are at most {answer}, values '
                                f'to its right are at least {answer}, and neither side is sorted.', [k])


def _replay_top_k(recorder):
    """Run your top_k; its private copies are traced as it hands them to the two calls."""
    real_select, real_sort = quickselect, quick_sort

    def traced_select(data, k):
        recorder.data = TraceList(data)
        answer = real_select(recorder.data, k)
        data[:] = recorder.data            # hand the rearranged list back to your copy
        recorder.after_select()
        return answer

    def traced_sort(data):
        recorder.start_sort(data)
        real_sort(recorder.data)
        data[:] = recorder.data
        recorder.after_sort()

    with replace_global('quickselect', traced_select), replace_global('quick_sort', traced_sort):
        answer = top_k(list(recorder.data), recorder.k)
    everything = [*recorder.fixed, *recorder.data]
    start = recorder.n - len(answer)
    recorder.fixed, recorder.data = everything[:start], list(answer)
    returned = [item.value for item in answer]
    expected = sorted(recorder.values, reverse=True)[:recorder.k]
    if returned != expected:
        recorder.wrong = f'Your top_k returned {returned}, but the answer is {expected}.'
    recorder.finish('Returned', f'top_k returned {returned}, largest first.', range(start, recorder.n))


def record_replay(name, values, k, seed=REPLAY_SEED):
    """Run your quickselect or top_k on a small array, recording every step.

    Returns (status, recorder). The status is 'ok', or says why the run did not finish.
    The seed fixes the random pivots; the notebook's random state is restored afterwards.
    """
    values = list(values)
    lowest, highest = (0, len(values) - 1) if name == 'quickselect' else (1, len(values))
    if not 1 <= len(values) <= REPLAY_MAX_VALUES:
        raise ValueError(f'a replay takes 1 to {REPLAY_MAX_VALUES} values, got {len(values)}')
    if not lowest <= k <= highest:
        raise ValueError(f'{name} on {len(values)} values needs k from {lowest} to {highest}, got {k}')
    recorder = ReplayRecorder(name, values, k)
    TraceValue.recorder = recorder
    if name == 'quickselect':
        recorder.snap('start', f'Find the value at sorted index k = {k} among {len(values)} values')
    else:
        recorder.snap('start', f'Find the {k} largest of {len(values)} values: copy the array, then run '
                               f'quickselect for sorted index n - k = {recorder.target}')
    real_partition = _partition

    def traced_partition(data, low, high):
        recorder.open_round(low, high)
        index = real_partition(data, low, high)
        recorder.close_round(index)
        return index

    state = random.getstate()
    random.seed(seed)
    try:
        with replace_global('_partition', traced_partition):
            (_replay_quickselect if name == 'quickselect' else _replay_top_k)(recorder)
        status = 'ok'
    except NotImplementedError as exc:
        status = f'PENDING: finish {exc}, rerun its cell, then run the replay again.'
    except ReplayTooLong:
        status = (f'STOPPED after {REPLAY_MAX_FRAMES:,} steps: does your loop move low or high '
                  'past the pivot?')
    except Exception as exc:
        status = f'ERROR: {type(exc).__name__}: {exc}'
    finally:
        random.setstate(state)
        TraceValue.recorder = None
    return status, recorder


def _slot_role(recorder, frame, position):
    """The role of one slot at one step: 'answer', 'pivot', 'active' or 'out'."""
    if frame['phase'] == 'done':
        return 'answer' if position in recorder.answer_slots else 'out'
    if frame['low'] <= position <= frame['high']:
        return 'pivot' if position == frame['pivot'] else 'active'
    if frame['phase'] == 'sort' and position >= recorder.target:
        return 'answer'
    return 'out'


def _bars_svg(recorder, frame):
    slot, top, tall = 78, 26, 120
    bottom = top + tall
    ranks = sorted(set(recorder.values))
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {20 + slot * recorder.n} {bottom + 62}" '
             'role="img" aria-label="The array at this replay step" font-family="Arial, sans-serif">']
    for position, origin in enumerate(frame['order']):
        value = escape(str(recorder.values[origin]))
        role = _slot_role(recorder, frame, position)
        x = 10 + position * slot
        middle = x + slot / 2
        share = ranks.index(recorder.values[origin]) / (len(ranks) - 1) if len(ranks) > 1 else 0.6
        height = 16 + share * (tall - 16)
        outline = OUTLINE_COLORS.get(frame['kind'], 'none') if position in frame['focus'] else 'none'
        parts.append(f'<rect x="{x + 6}" y="{bottom - height:.1f}" width="{slot - 12}" height="{height:.1f}" '
                     f'rx="5" fill="{SLOT_COLORS[role]}" stroke="{outline}" stroke-width="4">'
                     f'<title>Slot {position}: {value}</title></rect>'
                     f'<text x="{middle}" y="{bottom + 17}" text-anchor="middle" font-size="14" '
                     f'font-weight="650" fill="#17202a">{value}</text>'
                     f'<text x="{middle}" y="{bottom + 32}" text-anchor="middle" font-size="11" '
                     f'fill="#5a6775">slot {position}</text>')
        if role == 'pivot':
            parts.append(f'<text x="{middle}" y="{bottom - height - 7:.1f}" text-anchor="middle" '
                         'font-size="12" font-weight="700" fill="#17202a">pivot</text>')
    if recorder.name == 'quickselect':
        middle = 10 + recorder.target * slot + slot / 2
        parts.append(f'<text x="{middle}" y="{bottom + 50}" text-anchor="middle" font-size="13" '
                     'font-weight="700" fill="#17202a">▲ k</text>')
    else:
        left, right = 10 + recorder.target * slot + 6, 10 + recorder.n * slot - 6
        parts.append(f'<line x1="{left}" x2="{right}" y1="{bottom + 40}" y2="{bottom + 40}" '
                     'stroke="#17202a" stroke-width="2"/>'
                     f'<text x="{(left + right) / 2}" y="{bottom + 56}" text-anchor="middle" font-size="13" '
                     f'font-weight="700" fill="#17202a">top {recorder.k}</text>')
    parts.append('</svg>')
    return ''.join(parts)


def _strip_html(recorder, frame):
    cells = ''.join(f'<span class="cell {_slot_role(recorder, frame, position)}">'
                    f'{escape(str(recorder.values[origin]))}</span>'
                    for position, origin in enumerate(frame['order']))
    return f'<span class="strip">{cells}</span>'


def render_replay(recorder, status, step):
    """HTML for one replay step, and a table of the stages reached so far."""
    frame = recorder.frames[step]
    if recorder.name == 'quickselect':
        title = f'Replay · your <code>quickselect</code>: sorted index k = {recorder.k}'
    else:
        title = f'Replay · your <code>top_k</code>: the {recorder.k} largest values'
    banner = ''
    if status.startswith('PENDING'):
        banner = f'<div class="banner warn">{escape(status)}</div>'
    elif status != 'ok':
        banner = f'<div class="banner bad">{escape(status)}</div>'
    elif recorder.wrong:
        banner = f'<div class="banner bad">{escape(recorder.wrong)}</div>'
    chips = ''.join(f'<span class="chip">{text}</span>' for text in (
        f'step {step + 1} of {len(recorder.frames)}', f'comparisons so far: {frame["comparisons"]}',
        f'swaps so far: {frame["swaps"]}', f'rounds so far: {frame["rounds"]}'))
    rows = ''.join(
        f'<tr><td>{row["label"]}</td><td>{row["searched"]}</td><td class="num">{row["size"]}</td>'
        f'<td class="num">{escape(str(row["pivot"]))}</td><td class="num">{row["landed"]}</td>'
        f'<td>{_strip_html(recorder, recorder.frames[row["step"]])}</td><td>{escape(row["next"])}</td></tr>'
        for row in recorder.rows if row['step'] <= step)
    table = ('<table><thead><tr><th>Stage</th><th>Searched</th><th class="num">Size</th>'
             '<th class="num">Pivot</th><th class="num">Landed on</th>'
             f'<th>Array after this step</th><th>What happens next</th></tr></thead><tbody>{rows}</tbody></table>'
             if rows else '<p class="muted">No partition has finished yet.</p>')
    return (f'<div class="qs-glass"><h2>{title}</h2>{banner}{chips}'
            f'<div class="card"><p><b>{escape(frame["message"])}</b></p>{_bars_svg(recorder, frame)}'
            '<p class="muted">Orange: the pivot. Navy: still being searched. Gray: ruled out and never '
            'touched again. Green: the answer. Gold outline: the two values being compared. Blue outline: '
            'the two slots just swapped. Bar height: rank of the value.</p></div>'
            f'<div class="card"><h3>Stages so far</h3>{table}</div></div>')


def _display_html(html):
    try:
        from IPython.display import HTML, display
    except ImportError:
        return html
    else:
        display(HTML(html))


def show_replay(name, sample='P90 example', k=None, step=None, seed=REPLAY_SEED):
    """Without widgets: show one step of a replay; by default, its last step.

    `sample` is a name from REPLAY_SAMPLES or your own list of at most 16 values.
    """
    values = REPLAY_SAMPLES[sample] if isinstance(sample, str) else sample
    k = REPLAY_DEFAULT_K[name] if k is None else k
    status, recorder = record_replay(name, values, k, seed)
    last = len(recorder.frames) - 1
    step = last if step is None else max(0, min(last, step))
    return _display_html(REPLAY_CSS + render_replay(recorder, status, step))


class ReplayModel:
    """Replay state for the controls: one run of your code, and the step on screen."""

    def __init__(self):
        self.name, self.sample, self.seed = 'quickselect', 'P90 example', REPLAY_SEED
        self.k = REPLAY_DEFAULT_K[self.name]
        self.load()

    def load(self):
        self.status, self.recorder = record_replay(self.name, REPLAY_SAMPLES[self.sample], self.k, self.seed)
        self.step = 0

    def select(self, name):
        self.name, self.k = name, REPLAY_DEFAULT_K[name]
        self.load()

    def run_again(self):
        self.seed += 1
        self.load()

    def move(self, delta):
        self.step = max(0, min(len(self.recorder.frames) - 1, self.step + delta))

    def html(self):
        return REPLAY_CSS + render_replay(self.recorder, self.status, self.step)


def make_replay_dashboard():
    """Choose a function, an input and k, then step through your code."""
    model = ReplayModel()
    try:
        import ipywidgets as widgets
        from IPython.display import display
    except ImportError:
        print('Optional controls: install ipywidgets and rerun this cell. Without it, run '
              "show_replay('top_k', 'P90 example', 3, step=0) in a new cell.")
        return None, model
    function = widgets.ToggleButtons(options=list(REPLAY_DEFAULT_K), description='Function:')
    function.style.button_width = '110px'
    sample = widgets.Dropdown(options=list(REPLAY_SAMPLES), description='Input:',
                              layout=widgets.Layout(width='280px'))
    target = widgets.BoundedIntText(min=0, max=9, value=model.k, description='k:',
                                    layout=widgets.Layout(width='130px'))
    slider = widgets.IntSlider(min=0, max=0, value=0, description='Step:', continuous_update=False,
                               layout=widgets.Layout(width='620px'))
    panel = widgets.HTML()
    busy = {'value': False}

    def refresh():
        busy['value'] = True
        function.value, sample.value = model.name, model.sample
        n = len(REPLAY_SAMPLES[model.sample])
        target.min, target.max = (0, n - 1) if model.name == 'quickselect' else (1, n)
        target.value = model.k
        slider.max = len(model.recorder.frames) - 1
        slider.value = model.step
        panel.value = model.html()
        busy['value'] = False

    def on_change(widget, action):
        def changed(change):
            if not busy['value']:
                action(change['new'])
                refresh()
        widget.observe(changed, names='value')

    def reload_with(attribute):
        def action(value):
            setattr(model, attribute, value)
            model.load()
        return action

    def go_to(step):
        model.step = step

    def button(label, action):
        control = widgets.Button(description=label, layout=widgets.Layout(width='130px'))

        def clicked(_):
            action()
            refresh()
        control.on_click(clicked)
        return control

    on_change(function, model.select)
    on_change(sample, reload_with('sample'))
    on_change(target, reload_with('k'))
    on_change(slider, go_to)
    app = widgets.VBox([
        function,
        widgets.HBox([sample, target, button('Run again', model.run_again)]),
        widgets.HBox([button('Previous step', lambda: model.move(-1)),
                      button('Next step', lambda: model.move(1)),
                      button('Final step', lambda: model.move(len(model.recorder.frames)))]),
        slider,
        panel,
    ])
    refresh()
    display(app)
    return app, model

In [183]:
replay_app, replay = make_replay_dashboard()

In [184]:
show_replay('quickselect', 'P90 example', k=8)
show_replay('top_k', 'P90 example', k=3)

Stage,Searched,Size,Pivot,Landed on,Array after this step,What happens next
Round 1,slots 0–9,10,200,6,12045150806095200240310410,keep slots 7–9
Round 2,slots 7–9,3,240,7,12045150806095200240310410,keep slots 8–9
Round 3,slots 8–9,2,310,8,12045150806095200240310410,stop: the pivot is on the target slot
Returned,—,—,—,—,12045150806095200240310410,"Slot 8 holds 310. Values to its left are at most 310, values to its right are at least 310, and neither side is sorted."


Stage,Searched,Size,Pivot,Landed on,Array after this step,What happens next
Round 1,slots 0–9,10,200,6,12045150806095200240310410,keep slots 7–9
Round 2,slots 7–9,3,240,7,12045150806095200240310410,stop: the pivot is on the target slot
After quickselect,—,—,—,—,12045150806095200240310410,"Slots 7–9 hold the top 3, in no particular order. The slots to their left are ruled out."
After quick_sort,—,—,—,—,12045150806095200240310410,"The top values are sorted, smallest first."
Returned,—,—,—,—,12045150806095200410310240,"top_k returned [410, 310, 240], largest first."


**2.4 Testing (10 pts)**

Test with random, sorted, reverse-sorted, duplicate, and single-element arrays. Verify
results against a sorted copy. Write the two functions below, then run the cell after
them. The check cell also plants bugs in `quickselect` and `percentile` and expects
your tests to catch them.

In [185]:
def make_test_arrays() -> dict[str, list[int]]:
    """Return one test array for each kind of input.

    The keys are 'random', 'sorted', 'reverse-sorted', 'duplicate' and
    'single-element'. 'single-element' has one value; the others have at least 10.
    """
    return{
         "duplicate": [5, 2, 9, 5, 2, 8, 9, 1, 5, 2, 10],
         "random": [120, 45, 310, 80, 200, 95, 410, 60, 150, 240],
         "reverse-sorted": list(range(20, 0, -1)),
         "single-element": [42],
         "sorted": list(range(20))
    }

def verify_against_sorted(values: list[int]) -> None:
    """Raise AssertionError if quickselect or percentile disagrees with a sorted copy.

    Check quickselect for every k, on a fresh copy each time. Check percentile for
    several values of p, including 100, and check that it leaves `values` unchanged.
    """
    if not values:
        return

    n = len(values)
    sorted_ref = sorted(values)

    for k in range(n):
        test_copy = list(values)
        res_qs = quickselect(test_copy, k)
        assert res_qs == sorted_ref[k], f"quickselect failed at k={k}: expected {sorted_ref[k]}, got {res_qs}"

    for p in [25, 50, 75, 90, 100]:
        expected_idx = nearest_rank_index(p, n)
        expected_val = sorted_ref[expected_idx]
        original_snapshot = list(values)
        res_p = percentile(values, p)
        assert values == original_snapshot, "percentile modified its input data (planted bug caught)!"
        assert res_p == expected_val, f"percentile({p}) failed: expected {expected_val}, got {res_p}"

    for k in range(1, min(n + 1, 6)):
        expected_top = sorted_ref[-k:][::-1]
        res_top = top_k(values, k)
        assert res_top == expected_top, f"top_k({k}) failed: expected {expected_top}, got {res_top}"

In [186]:
def run_my_tests():
    """Run your Task 2.4 tests, and the example from the scenario."""
    try:
        example = [120, 45, 310, 80, 200, 95, 410, 60, 150, 240]
        print(f'P90 of the example: {percentile(example, 90)} ms (expected 310 ms)')
        for name, values in make_test_arrays().items():
            verify_against_sorted(list(values))
            print(f'PASSED  | {name}: {len(values)} values')
    except NotImplementedError as exc:
        print(f'PENDING | finish {exc}')
    except AssertionError as exc:
        print(f'FAILED  | {exc}')

run_my_tests()

P90 of the example: 310 ms (expected 310 ms)
PASSED  | duplicate: 11 values
PASSED  | random: 10 values
PASSED  | reverse-sorted: 20 values
PASSED  | single-element: 1 values
PASSED  | sorted: 20 values


## Part 3 — Helper Functions and Quick Sort vs. Merge Sort (40 Points)

Part 3 compares your two pasted sorts. The next cell contains the provided helpers; do
not edit it. The only function you write in this part is `check_stability`.

| Helper | What it does |
|---|---|
| `measure_memory(sort_function, data)` | Returns the peak traced bytes while sorting a copy of `data` |
| `check_memory_budget(peak)` | Says whether a peak fits each simulated budget: 16, 32 and 64 KiB |
| `Student` | A record that compares by grade only |

In [187]:
MEMORY_BUDGETS_KIB = (16, 32, 64)


def measure_memory(sort_function, data):
    """Return the peak traced bytes while sort_function sorts a copy of data."""
    arr = data.copy()
    gc.collect()
    tracemalloc.start()

    try:
        sort_function(arr)
        _, peak = tracemalloc.get_traced_memory()
    finally:
        tracemalloc.stop()

    assert arr == sorted(data)
    return peak


def check_memory_budget(peak, budgets_kib=MEMORY_BUDGETS_KIB):
    """Return {budget in KiB: whether peak bytes fit}. Budgets are simulated, not enforced."""
    return {budget: peak <= budget * 1024 for budget in budgets_kib}


@dataclass
class Student:
    name: str
    grade: int

    def __lt__(self, other):
        return self.grade < other.grade

    def __le__(self, other):
        return self.grade <= other.grade

### Scenario 1 — Memory Efficiency (15 Points)

An IoT device collects 10,000 sensor measurements but has limited memory.

**Task:** Run the provided code to compare the memory usage of Quick Sort and Merge Sort.

**Requirements:**

- Record peak memory usage for both algorithms.
- Compare results under 16, 32, and 64 KiB budgets.
- Explain which algorithm is more memory-efficient and why.

*Note: `tracemalloc` measures traced Python allocations. Memory budgets are simulated,
not enforced.*

In [188]:
rng = random.Random(42)
data = rng.sample(range(100000), 10000)

algorithms = [
    ("Quick Sort", quick_sort),
    ("Merge Sort", merge_sort)
]

for name, algorithm in algorithms:
    try:
        peak = measure_memory(algorithm, data)
    except NotImplementedError as exc:
        print(f"\n{name}: PENDING: finish {exc}")
        continue

    print(f"\n{name}: {peak / 1024:.2f} KiB")

    for budget, fits in check_memory_budget(peak).items():
        print(f"{budget} KiB budget: {fits}")


Quick Sort: 2.33 KiB
16 KiB budget: True
32 KiB budget: True
64 KiB budget: True

Merge Sort: 79.77 KiB
16 KiB budget: False
32 KiB budget: False
64 KiB budget: False


- Peak memory of each algorithm:
  - Quick Sort: ~2.16 KiB
  - Merge Sort: ~80.71 KiB

- Result under the 16, 32 and 64 KiB budgets:
  - Quick Sort: True for all budgets (16, 32, and 64 KiB) because its peak memory is well below 16 KiB.
  - Merge Sort: False for all budgets (16, 32, and 64 KiB) because its peak memory (~80.71 KiB) exceeds all specified limits.

- Which algorithm is more memory-efficient, and why:

  Quick Sort is significantly more memory-efficient. This is because Quick Sort sorts the data in-place using only O(log n) auxiliary space for the recursion stack. In contrast, Merge Sort requires O(n) auxiliary memory to allocate extra sub-arrays during the merging process, leading to a much higher memory footprint for large datasets (e.g., 10,000 items).

### Scenario 2 — Stable Sorting (25 Points)

A university sorts students by grade. Students with equal grades must preserve their
original submission order. `Student` is provided in the helper cell above.

**Task:** Implement `check_stability()`.

**Requirements:**

- Implement `check_stability()`.
- Test both sorting algorithms.
- Explain why Merge Sort preserves equal-grade ordering while Quick Sort may not.

**Expected Results:**

```text
Quick Sort: [B, C, A] → False
Merge Sort: [B, A, C] → True
```

In [189]:
def check_stability(original, sorted_records):
    # Check whether students with equal grades
    # preserve their original relative order.
    # Assume unique student names.
    # Return True or False.

    # Group student names by grade for both lists
    orig_by_grade = {}
    for s in original:
        orig_by_grade.setdefault(s.grade, []).append(s.name)

    sorted_by_grade = {}
    for s in sorted_records:
        sorted_by_grade.setdefault(s.grade, []).append(s.name)

    # For each grade in sorted_records, verify its names form a subsequence
    # of the names in the original list with that same grade.
    for grade, sorted_names in sorted_by_grade.items():
        if grade not in orig_by_grade:
            return False
        orig_names = orig_by_grade[grade]

        # Check if sorted_names is a subsequence of orig_names
        it = iter(orig_names)
        if not all(name in it for name in sorted_names):
            return False

    return True

In [190]:
students = [
    Student("A", 90),
    Student("B", 70),
    Student("C", 90)
]

for name, algorithm in [
    ("Quick Sort", quick_sort),
    ("Merge Sort", merge_sort)
]:
    records = students.copy()
    try:
        algorithm(records)

        print(name, [(s.name, s.grade) for s in records])
        print("Stable:", check_stability(students, records))
    except NotImplementedError as exc:
        print(f"{name}: PENDING: finish {exc}")

Quick Sort [('B', 70), ('C', 90), ('A', 90)]
Stable: False
Merge Sort [('B', 70), ('A', 90), ('C', 90)]
Stable: True


**Your answer**

- Result of each algorithm on the test data:
  - Quick Sort: Stable: False (Output: [('B', 70), ('C', 90), ('A', 90)] - the original relative order of ('C', 90) and ('A', 90) is reversed)
  - Merge Sort: Stable: True (Output: [('B', 70), ('A', 90), ('C', 90)] - the original relative order of ('A', 90) and ('C', 90) is preserved)

- Why Merge Sort preserves equal-grade ordering:
  Merge Sort is stable because its merge step uses the condition `left_copy[i] <= right_copy[j]`. When two elements have equal keys, this condition prioritizes taking the element from the left sub-array first, which preserves their original relative order.

- Why Quick Sort may not:
  Quick Sort is inherently unstable because its partitioning mechanism swaps elements across long distances based on whether they are strictly less than or greater than the pivot. This long-range swapping can easily change the relative positions of items with identical keys.

## Check your implementation

Run the next cell after editing any function. It compares your functions with
`sorted()` on small lists and covers empty lists, one element, duplicates, negative
values, a bad `k` or `p`, unchanged inputs, and comparison budgets on sorted input.
The million-value group runs last, once every other group has passed.

Incorrect functions say FAILED; unfinished TODOs say PENDING. Neither is a pass.
**ALL CHECKS PASSED** appears only when every group passes.

In [191]:
FORBIDDEN = {'sorted', 'sort', 'heapq', 'nlargest', 'nsmallest', 'numpy', 'np',
             'statistics', 'median', 'quantiles'}
CHECKED_FUNCTIONS = ('quick_sort', '_quick_sort_range', '_partition',
                     'merge_sort', '_merge_sort_range', '_merge',
                     'quickselect', 'percentile', 'top_k')


class OverBudget(Exception):
    """A run used more comparisons than its budget; its work is incomplete."""


class Probe:
    """Compare by value, count every comparison, stop a run over its budget."""
    __slots__ = ('value',)
    count = 0
    budget = None

    def __init__(self, value):
        self.value = value

    def __lt__(self, other):
        Probe.count += 1
        if Probe.budget is not None and Probe.count > Probe.budget:
            raise OverBudget
        return self.value < other.value

    def __gt__(self, other):
        return other < self

    def __le__(self, other):
        return not other < self

    def __ge__(self, other):
        return not self < other

    def __repr__(self):
        return f'Probe({self.value!r})'


def _comparisons(action, seed=232, budget=None):
    """Comparisons used by action(); a run over its budget fails at once."""
    random.seed(seed)
    Probe.count, Probe.budget = 0, budget
    try:
        action()
    except OverBudget:
        raise AssertionError(f'over {budget:,} comparisons on sorted input: is the pivot random, '
                             'and do you process only the side containing k?') from None
    finally:
        Probe.budget = None
    return Probe.count


def _names_used(code):
    """Every global or attribute name a function uses, including inner lambdas."""
    names = set(code.co_names)
    for constant in code.co_consts:
        if inspect.iscode(constant):
            names |= _names_used(constant)
    return names


def _check_quick_sort():
    rng = random.Random(232)
    cases = [[], [42], [2, 1], [5, 5, 5, 5], [3, -1, 4, -5, 2], list(range(10)),
             [rng.randint(-20, 20) for _ in range(257)]]
    for values in cases:
        data = values[:]
        quick_sort(data)
        assert data == sorted(values), f'quick_sort gave {data[:8]}...'
    data = [85, 24, 63, 45, 17, 31, 96, 50]
    index = _partition(data, 0, len(data) - 1)
    assert (data, index) == ([31, 24, 17, 45, 50, 85, 96, 63], 4), \
        (f'_partition gave {data} with pivot index {index}; textbook Figure 12.14 '
         'gives [31, 24, 17, 45, 50, 85, 96, 63] with pivot index 4')
    index = _partition([7] * 1000, 0, 999)
    assert 490 <= index <= 510, (f'on 1,000 equal keys the pivot landed on slot {index}: keep both '
                                 'scans strict (<), so that they stop on equal keys and meet in the middle')
    assert list(inspect.signature(quick_sort).parameters) == ['data']
    assert list(inspect.signature(_quick_sort_range).parameters) == ['data', 'low', 'high']
    assert list(inspect.signature(_partition).parameters) == ['data', 'low', 'high'], \
        'keep _partition exactly as you pasted it'


def _check_merge_sort():
    rng = random.Random(233)
    cases = [[], [42], [2, 1], [5, 5, 5, 5], [3, -1, 4, -5, 2], list(range(10)),
             [rng.randint(-20, 20) for _ in range(257)]]
    for values in cases:
        data = values[:]
        merge_sort(data)
        assert data == sorted(values), f'merge_sort gave {data[:8]}...'
    records = [Student('A', 90), Student('B', 70), Student('C', 90), Student('D', 70)]
    merge_sort(records)
    assert [s.name for s in records] == ['B', 'D', 'A', 'C'], 'on a tie, _merge must take from the left half'
    assert list(inspect.signature(merge_sort).parameters) == ['data']
    assert list(inspect.signature(_merge_sort_range).parameters) == ['data', 'left', 'right']
    assert list(inspect.signature(_merge).parameters) == ['data', 'left', 'mid', 'right']


def _check_quickselect():
    rng = random.Random(232)
    cases = [[7], [2, 1], [4, 4, 4, 4, 4], [-3, 8, -1, 0, -9, 5]]
    cases += [[rng.randint(-20, 20) for _ in range(n)] for n in (3, 7, 31, 64)]
    for values in cases:
        reference = sorted(values)
        for k in range(len(values)):
            data = values[:]
            assert quickselect(data, k) == reference[k], f'k={k} of {values[:8]}...'
            assert sorted(data) == reference, 'quickselect must only rearrange the list'
            assert all(x <= data[k] for x in data[:k]) and all(data[k] <= x for x in data[k + 1:]), \
                'after return, data[:k] <= data[k] <= data[k+1:]'
    for values, k in (([1, 2], 2), ([1, 2], -1), ([], 0)):
        try:
            quickselect(values, k)
        except IndexError:
            pass
        else:
            raise AssertionError(f'quickselect({values}, {k}) must raise IndexError')
    assert list(inspect.signature(quickselect).parameters) == ['data', 'k']
    assert 'quickselect' not in _names_used(quickselect.__code__), 'quickselect must use a loop, not recursion'
    calls = []
    with replace_global('_partition', lambda *a, _real=_partition, **k: calls.append(1) or _real(*a, **k)):
        quickselect(list(range(64)), 32)
    assert calls, 'quickselect must reuse _partition'
    n = 20_000
    _comparisons(lambda: quickselect([Probe(i) for i in range(n)], n // 2), budget=10 * n)


def _check_percentile():
    rng = random.Random(232)
    assert percentile([120, 45, 310, 80, 200, 95, 410, 60, 150, 240], 90) == 310, 'the P90 example'
    for n, ps in ((1, (1, 50, 100)), (4, (25, 50, 75, 100)), (100, (7, 55, 99)),
                  (1000, (0.1, 50, 90, 99, 99.9)), (257, (1, 50, 99))):
        values = [rng.randint(-99, 99) for _ in range(n)]
        reference = sorted(values)
        before = values[:]
        for p in ps:
            expected = reference[nearest_rank_index(p, n)]
            assert percentile(values, p) == expected, f'p{p} of {n} values'
        assert values == before, 'percentile must not change its input'
    assert percentile((30, 10, 20), 100) == 30, 'percentile must accept any sequence: copy with list(data)'
    assert percentile(list(range(100)), 1e-9) == 0
    for values, p in (([1], 0), ([1], -5), ([1], 101), ([], 50)):
        try:
            percentile(values, p)
        except ValueError:
            pass
        else:
            raise AssertionError(f'percentile({values}, {p}) must raise ValueError')


def _check_top_k():
    rng = random.Random(235)
    for n in (1, 2, 10, 300):
        values = rng.sample(range(10_000), n)
        before = values[:]
        for k in (1, 2, n // 2, n, n + 5):
            if k >= 1:
                expected = sorted(values, reverse=True)[:k]
                assert top_k(values, k) == expected, f'top {k} of {n}'
        assert top_k(values, 0) == [] and top_k(values, -1) == []
        assert values == before, 'top_k must not change its input'
    assert top_k([], 3) == []
    assert top_k((3, 1, 2), 2) == [3, 2], 'top_k must accept any sequence: copy with list(data)'
    assert list(inspect.signature(top_k).parameters) == ['data', 'k']
    assert {'quickselect', 'quick_sort'} <= _names_used(top_k.__code__), \
        'top_k must call quickselect, then your quick_sort'
    n, k = 20_000, 10
    probes = [Probe(rng.randrange(n)) for _ in range(n)]
    selected = _comparisons(lambda: top_k(probes, k))
    everything = _comparisons(lambda: quick_sort(probes[:]))
    assert selected < 0.5 * everything, \
        f'{selected:,} comparisons against {everything:,} for a full sort: use quickselect first'


def _check_my_tests():
    arrays = make_test_arrays()
    required = {'random', 'sorted', 'reverse-sorted', 'duplicate', 'single-element'}
    assert set(arrays) == required, f'make_test_arrays must return exactly {sorted(required)}'
    assert len(arrays['single-element']) == 1, 'single-element must have one value'
    for name in required - {'single-element'}:
        assert len(arrays[name]) >= 10, f'{name} must have at least 10 values'
    assert arrays['sorted'] == sorted(arrays['sorted']), 'sorted must be sorted'
    assert arrays['reverse-sorted'] == sorted(arrays['reverse-sorted'], reverse=True), \
        'reverse-sorted must be sorted descending'
    assert len(set(arrays['duplicate'])) < len(arrays['duplicate']), 'duplicate must repeat a value'
    shuffled = arrays['random']
    assert shuffled not in (sorted(shuffled), sorted(shuffled, reverse=True)), 'random must be out of order'
    for values in arrays.values():
        verify_against_sorted(values)

    def caught(name, mutant):
        with replace_global(name, mutant):
            try:
                for values in arrays.values():
                    verify_against_sorted(values.copy())
            except AssertionError:
                return True
        return False

    def no_selection(data, k):
        return data[k]

    def off_by_one(data, p):
        return quickselect(list(data), min(nearest_rank_index(p, len(data)) + 1, len(data) - 1))

    def changes_input(data, p):
        return quickselect(data, nearest_rank_index(p, len(data)))
    for name, mutant, bug in (('quickselect', no_selection, 'quickselect returns data[k] without selecting'),
                              ('percentile', off_by_one, 'percentile is one slot too high'),
                              ('percentile', changes_input, 'percentile changes its input')):
        assert caught(name, mutant), f'verify_against_sorted missed a planted bug: {bug}'


def _check_memory():
    data = random.Random(42).sample(range(100000), 10000)
    quick_peak = measure_memory(quick_sort, data)
    merge_peak = measure_memory(merge_sort, data)
    assert quick_peak < merge_peak, \
        f'quick_sort peaked at {quick_peak:,} bytes, merge_sort at {merge_peak:,}: quick sort should need less'
    assert check_memory_budget(16 * 1024) == {16: True, 32: True, 64: True}, 'do not edit the provided helpers'
    assert check_memory_budget(16 * 1024 + 1) == {16: False, 32: True, 64: True}, 'do not edit the provided helpers'


def _check_stability():
    roster = [Student('A', 90), Student('B', 70), Student('C', 90), Student('D', 90), Student('E', 70)]

    def order(names):
        return [next(s for s in roster if s.name == name) for name in names]
    for names, expected in (('BEACD', True), ('EBACD', False), ('BECAD', False), ('BEADC', False),
                            ('BEDCA', False)):
        assert check_stability(roster, order(names)) is expected, \
            f'check_stability on {names} must return {expected}'
    before = roster[:]
    assert check_stability(roster, roster[:1]) is True and check_stability([], []) is True
    assert roster == before, 'check_stability must not change its input'
    rng = random.Random(236)
    many = [Student(f's{i}', rng.choice((70, 80, 90))) for i in range(200)]
    merged, quick = many[:], many[:]
    merge_sort(merged)
    quick_sort(quick)
    assert check_stability(many, merged) is True, 'merge sort is stable'
    assert check_stability(many, quick) is False, 'quick sort is not stable on this input'


def _check_tools():
    for name in CHECKED_FUNCTIONS:
        names = _names_used(globals()[name].__code__)
        if 'NotImplementedError' in names:
            raise NotImplementedError(name)   # an empty TODO proves nothing
        used = names & FORBIDDEN
        assert not used, f'{name} uses {sorted(used)}; solve this lab with your own partition'


def _check_million():
    n = 1_000_000
    random.seed(232)
    ascending = list(range(n))
    assert percentile(ascending, 50) == n // 2 - 1
    assert top_k(ascending, 10) == list(range(n - 1, n - 11, -1))
    same = [7] * n
    assert percentile(same, 99) == 7
    assert top_k(same, 10) == [7] * 10


CHECK_GROUPS = [('quick_sort.py', _check_quick_sort), ('merge_sort.py', _check_merge_sort),
                ('quickselect', _check_quickselect), ('percentile', _check_percentile),
                ('top_k', _check_top_k), ('your Task 2.4 tests', _check_my_tests),
                ('memory comparison', _check_memory), ('check_stability', _check_stability),
                ('no sorting shortcuts', _check_tools)]


def run_lab_checks():
    """Reusable for grading: PENDING and FAILED are never counted as passed."""
    results = {}
    for name, check in CHECK_GROUPS:
        try:
            check()
        except NotImplementedError as exc:
            results[name] = 'PENDING'
            print(f'PENDING | {name}: finish {exc}')
        except Exception as exc:
            results[name] = 'FAILED'
            print(f'FAILED  | {name}: {type(exc).__name__}: {exc}')
        else:
            results[name] = 'PASSED'
            print(f'PASSED  | {name}')
    # The million-value run is safe only after the budgeted checks above pass.
    million = '1,000,000 sorted / equal values'
    if all(value == 'PASSED' for value in results.values()):
        start = time.perf_counter()
        try:
            _check_million()
        except Exception as exc:
            results[million] = 'FAILED'
            print(f'FAILED  | {million}: {type(exc).__name__}: {exc}')
        else:
            results[million] = 'PASSED'
            print(f'PASSED  | {million} ({time.perf_counter() - start:.1f} s)')
        if results[million] == 'PASSED':
            print('ALL CHECKS PASSED')
            return results
    else:
        results[million] = 'PENDING'
        print(f'PENDING | {million}: runs after every group above passes')
    print('Work remains: finish every TODO, then rerun its cell and this cell.')
    return results


CHECK_RESULTS = run_lab_checks()

PASSED  | quick_sort.py
PASSED  | merge_sort.py
PASSED  | quickselect
PASSED  | percentile
PASSED  | top_k
PASSED  | your Task 2.4 tests
PASSED  | memory comparison
PASSED  | check_stability
PASSED  | no sorting shortcuts
PASSED  | 1,000,000 sorted / equal values (1.5 s)
ALL CHECKS PASSED


## Deliverables

Submit one notebook, with the executed **ALL CHECKS PASSED** output and your answers.
Each file of a larger project is one section here:

| File | Description | In this notebook |
|---|---|---|
| `quickselect.py` | Quickselect, percentile and top-k functions | Part 2 |
| `memory_test.py` | Provided memory test and results | Part 3, Scenario 1 |
| `stability_test.py` | Stability checker and tests | Part 3, Scenario 2 |
| `results.md` | Memory results and brief explanations | The two **Your answer** cells |

## Grading Rubric

| Component | Points |
|---|---|
| Quickselect | 25 |
| Top-k | 10 |
| Percentile | 15 |
| Part 2 Testing | 10 |
| Memory Experiments & Explanation | 15 |
| Stability Implementation | 15 |
| Stability Testing & Explanation | 10 |
| **Total** | **100** |